In [100]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rest_of_season.config import (
    SEASON,
    CURRENT_WEEK,
    N_SIMULATIONS,
    RANDOM_SEED,
    CURRENT_OUTPUT_DIR
)

TARGET_WEEK = CURRENT_WEEK
ROS_OUTPUT_DIR = CURRENT_OUTPUT_DIR

rng = np.random.default_rng(RANDOM_SEED)

assert ROS_OUTPUT_DIR.exists(), (
    f"Missing ROS output directory: {ROS_OUTPUT_DIR}"
)

print("Season:", SEASON)
print("Snapshot week:", TARGET_WEEK)
print("Simulations:", f"{N_SIMULATIONS:,}")
print("Remaining-games input:", ROS_OUTPUT_DIR / "ros_game_projections.parquet")
print("Output directory:", ROS_OUTPUT_DIR)


Season: 2026
Snapshot week: 9
Simulations: 50,000
Remaining-games input: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_09\ros_game_projections.parquet
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_09


In [101]:
games = pd.read_parquet(
    ROS_OUTPUT_DIR / "ros_game_projections.parquet"
)

teams = pd.read_parquet(
    ROS_OUTPUT_DIR / "team_snapshot.parquet"
)

required_game_cols = [
    "game_id",
    "week",
    "home_team",
    "away_team",
    "home_win_probability"
]

required_team_cols = [
    "team",
    "wins",
    "losses",
    "ties"
]

assert set(required_game_cols).issubset(games.columns)
assert set(required_team_cols).issubset(teams.columns)

assert len(teams) == 32
assert teams["team"].is_unique
assert games["game_id"].is_unique
assert games["home_win_probability"].between(0, 1).all()

print("Remaining games:", len(games))
print("Teams:", len(teams))

display(
    teams[
        ["team", "wins", "losses", "ties"]
    ].sort_values("team")
)

Remaining games: 151
Teams: 32


,team,wins,losses,ties
27,ARI,3,5,0
19,ATL,3,5,0
16,BAL,4,4,0
0,BUF,6,1,0
26,CAR,3,4,0
6,CHI,6,2,0
13,CIN,5,2,0
28,CLE,5,3,0
17,DAL,3,5,0
3,DEN,6,2,0


In [102]:
teams = teams.sort_values("team").reset_index(drop=True)

team_names = teams["team"].tolist()

team_to_idx = {
    team: i for i, team in enumerate(team_names)
}

home_idx = games["home_team"].map(team_to_idx)
away_idx = games["away_team"].map(team_to_idx)

assert home_idx.notna().all()
assert away_idx.notna().all()

home_idx = home_idx.to_numpy(dtype=int)
away_idx = away_idx.to_numpy(dtype=int)

home_prob = games[
    "home_win_probability"
].to_numpy(dtype=float)

current_wins = teams["wins"].to_numpy(dtype=int)
current_losses = teams["losses"].to_numpy(dtype=int)
current_ties = teams["ties"].to_numpy(dtype=int)

remaining_counts = (
    pd.concat([
        games["home_team"],
        games["away_team"]
    ])
    .value_counts()
    .reindex(team_names, fill_value=0)
    .to_numpy()
)

completed_counts = (
    current_wins + current_losses + current_ties
)

assert np.all(
    completed_counts + remaining_counts == 17
), "Some teams do not have exactly 17 total games."

assert completed_counts.sum() + 2 * len(games) == 544

print("Schedule validation passed.")

Schedule validation passed.


In [103]:
n_teams = len(team_names)
n_games = len(games)

simulated_wins = np.broadcast_to(
    current_wins,
    (N_SIMULATIONS, n_teams)
).copy()

simulated_losses = np.broadcast_to(
    current_losses,
    (N_SIMULATIONS, n_teams)
).copy()

simulated_ties = np.broadcast_to(
    current_ties,
    (N_SIMULATIONS, n_teams)
).copy()

simulated_home_wins = np.empty(
    (N_SIMULATIONS, n_games),
    dtype=bool
)

for i in range(n_games):

    home_wins = (
        rng.random(N_SIMULATIONS) < home_prob[i]
    )

    simulated_home_wins[:, i] = home_wins

    simulated_wins[:, home_idx[i]] += home_wins
    simulated_losses[:, home_idx[i]] += ~home_wins

    simulated_wins[:, away_idx[i]] += ~home_wins
    simulated_losses[:, away_idx[i]] += home_wins

assert np.all(
    simulated_wins
    + simulated_losses
    + simulated_ties == 17
)

print("Completed", N_SIMULATIONS, "season simulations.")

Completed 50000 season simulations.


In [104]:
projected_wins = simulated_wins.mean(axis=0)
projected_losses = simulated_losses.mean(axis=0)
projected_ties = simulated_ties.mean(axis=0)

standings = pd.DataFrame({
    "team": team_names,
    "current_wins": current_wins,
    "current_losses": current_losses,
    "current_ties": current_ties,
    "projected_wins": projected_wins,
    "projected_losses": projected_losses,
    "projected_ties": projected_ties,
    "win_p10": np.percentile(
        simulated_wins, 10, axis=0
    ),
    "win_median": np.median(
        simulated_wins, axis=0
    ),
    "win_p90": np.percentile(
        simulated_wins, 90, axis=0
    ),
    "prob_10_wins": (
        simulated_wins >= 10
    ).mean(axis=0),
    "prob_12_wins": (
        simulated_wins >= 12
    ).mean(axis=0)
})

standings["projected_remaining_wins"] = (
    standings["projected_wins"]
    - standings["current_wins"]
)

standings = standings.sort_values(
    "projected_wins",
    ascending=False
).reset_index(drop=True)

display(standings.round(3))

,team,current_wins,current_losses,current_ties,projected_wins,projected_losses,projected_ties,win_p10,win_median,win_p90,prob_10_wins,prob_12_wins,projected_remaining_wins
0,BUF,6,1,0,12.711,4.289,0.0,11.0,13.0,15.0,0.985,0.801,6.711
1,SF,6,1,0,12.241,4.759,0.0,10.0,12.0,14.0,0.965,0.695,6.241
2,MIN,6,1,0,11.757,5.243,0.0,10.0,12.0,14.0,0.929,0.571,5.757
3,DEN,6,2,0,11.751,5.249,0.0,10.0,12.0,14.0,0.944,0.582,5.751
4,JAX,5,2,0,11.442,5.558,0.0,10.0,11.0,13.0,0.903,0.497,6.442
5,PIT,6,2,0,11.271,5.729,0.0,9.0,11.0,13.0,0.889,0.447,5.271
6,CIN,5,2,0,11.006,5.994,0.0,9.0,11.0,13.0,0.837,0.382,6.006
7,CHI,6,2,0,10.977,6.023,0.0,9.0,11.0,13.0,0.846,0.365,4.977
8,DET,5,2,0,10.724,6.276,0.0,9.0,11.0,13.0,0.792,0.309,5.724
9,KC,5,2,0,10.623,6.377,0.0,9.0,11.0,13.0,0.773,0.286,5.623


In [105]:
win_distribution = pd.DataFrame({
    "team": np.repeat(team_names, 18),
    "final_wins": np.tile(np.arange(18), n_teams),
    "probability": np.concatenate([
        np.bincount(
            simulated_wins[:, i],
            minlength=18
        ) / N_SIMULATIONS
        for i in range(n_teams)
    ])
})

assert np.allclose(
    win_distribution.groupby("team")[
        "probability"
    ].sum().to_numpy(),
    1.0
)

display(win_distribution.head(20))

,team,final_wins,probability
0,ARI,0,0.00000
1,ARI,1,0.00000
2,ARI,2,0.00000
3,ARI,3,0.00612
4,ARI,4,0.04764
5,ARI,5,0.14776
6,ARI,6,0.25494
7,ARI,7,0.26228
8,ARI,8,0.18070
9,ARI,9,0.07544


In [106]:
standings["season"] = SEASON
standings["snapshot_week"] = TARGET_WEEK

standings.to_csv(
    ROS_OUTPUT_DIR / "projected_standings.csv",
    index=False
)

standings.to_parquet(
    ROS_OUTPUT_DIR / "projected_standings.parquet",
    index=False
)

win_distribution.to_csv(
    ROS_OUTPUT_DIR / "final_win_distribution.csv",
    index=False
)

win_distribution.to_parquet(
    ROS_OUTPUT_DIR / "final_win_distribution.parquet",
    index=False
)

np.savez_compressed(
    ROS_OUTPUT_DIR / "simulation_results.npz",
    team_names=np.array(team_names, dtype=str),
game_ids=games["game_id"].to_numpy(dtype=str),
home_teams=games["home_team"].to_numpy(dtype=str),
away_teams=games["away_team"].to_numpy(dtype=str),
    home_wins=simulated_home_wins,
    wins=simulated_wins,
    losses=simulated_losses,
    ties=simulated_ties
)

print("Saved ROS simulation outputs.")

Saved ROS simulation outputs.


In [107]:
assert np.isclose(
    standings["projected_remaining_wins"].sum(),
    len(games)
)

assert np.all(
    standings["projected_wins"]
    >= standings["current_wins"]
)

assert np.all(
    standings["projected_wins"] <= 17
)

print("All simulation validations passed.")

All simulation validations passed.


In [108]:
saved = np.load(
    ROS_OUTPUT_DIR / "simulation_results.npz"
)

assert saved["home_wins"].shape == (
    N_SIMULATIONS,
    len(games)
)

assert saved["wins"].shape == (
    N_SIMULATIONS,
    32
)

assert np.all(
    saved["wins"] + saved["losses"] + saved["ties"] == 17
)

assert np.allclose(
    saved["wins"].mean(axis=0),
    projected_wins
)

print("Game-level simulations saved and validated.")

Game-level simulations saved and validated.


In [109]:
DIVISIONS = {
    "AFC East": ["BUF", "MIA", "NE", "NYJ"],
    "AFC North": ["BAL", "CIN", "CLE", "PIT"],
    "AFC South": ["HOU", "IND", "JAX", "TEN"],
    "AFC West": ["DEN", "KC", "LAC", "LV"],
    "NFC East": ["DAL", "NYG", "PHI", "WAS"],
    "NFC North": ["CHI", "DET", "GB", "MIN"],
    "NFC South": ["ATL", "CAR", "NO", "TB"],
    "NFC West": ["ARI", "LA", "SF", "SEA"]
}

team_info = pd.DataFrame([
    {
        "team": team,
        "division": division,
        "conference": division.split()[0]
    }
    for division, members in DIVISIONS.items()
    for team in members
])

assert len(team_info) == 32
assert team_info["team"].is_unique
assert set(team_info["team"]) == set(team_names)

display(team_info)

,team,division,conference
0,BUF,AFC East,AFC
1,MIA,AFC East,AFC
2,NE,AFC East,AFC
3,NYJ,AFC East,AFC
4,BAL,AFC North,AFC
5,CIN,AFC North,AFC
6,CLE,AFC North,AFC
7,PIT,AFC North,AFC
8,HOU,AFC South,AFC
9,IND,AFC South,AFC


In [110]:
completed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weekly"
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

completed_games = pd.read_parquet(completed_path)

completed_games = completed_games.loc[
    (completed_games["week"] < TARGET_WEEK)
    & completed_games["home_score"].notna()
    & completed_games["away_score"].notna()
].copy()

completed_games = completed_games.merge(
    team_info.rename(columns={
        "team": "home_team",
        "division": "home_division",
        "conference": "home_conference"
    }),
    on="home_team",
    how="left",
    validate="many_to_one"
).merge(
    team_info.rename(columns={
        "team": "away_team",
        "division": "away_division",
        "conference": "away_conference"
    }),
    on="away_team",
    how="left",
    validate="many_to_one"
)

completed_games["is_division_game"] = (
    completed_games["home_division"]
    == completed_games["away_division"]
)

completed_games["is_conference_game"] = (
    completed_games["home_conference"]
    == completed_games["away_conference"]
)

assert completed_games["game_id"].is_unique

print("Completed games:", len(completed_games))

Completed games: 121


In [111]:
future_games = games.merge(
    team_info.rename(columns={
        "team": "home_team",
        "division": "home_division",
        "conference": "home_conference"
    }),
    on="home_team",
    how="left",
    validate="many_to_one"
).merge(
    team_info.rename(columns={
        "team": "away_team",
        "division": "away_division",
        "conference": "away_conference"
    }),
    on="away_team",
    how="left",
    validate="many_to_one"
)

future_games["is_division_game"] = (
    future_games["home_division"]
    == future_games["away_division"]
)

future_games["is_conference_game"] = (
    future_games["home_conference"]
    == future_games["away_conference"]
)

assert future_games["game_id"].is_unique
assert future_games[
    ["home_division", "away_division"]
].notna().all().all()

print("Future games:", len(future_games))

Future games: 151


In [112]:
all_game_ids = pd.concat([
    completed_games["game_id"],
    future_games["game_id"]
])

assert all_game_ids.is_unique, (
    "Completed and future games overlap."
)

assert len(all_game_ids) == 272, (
    f"Expected 272 NFL games, found {len(all_game_ids)}."
)

assert len(completed_games) + len(future_games) == 272

print("All 272 regular-season games accounted for.")

All 272 regular-season games accounted for.


In [113]:
team_info.to_parquet(
    ROS_OUTPUT_DIR / "team_divisions.parquet",
    index=False
)

completed_games.to_parquet(
    ROS_OUTPUT_DIR / "completed_games_tiebreakers.parquet",
    index=False
)

future_games.to_parquet(
    ROS_OUTPUT_DIR / "future_games_tiebreakers.parquet",
    index=False
)

print("Saved all playoff simulation inputs.")

Saved all playoff simulation inputs.


In [114]:
saved = np.load(
    ROS_OUTPUT_DIR / "simulation_results.npz",
    allow_pickle=True
)

saved_game_ids = saved["game_ids"].astype(str)
saved_team_names = saved["team_names"].astype(str)

assert np.array_equal(
    saved_team_names,
    np.array(team_names)
)

future_games = (
    future_games
    .set_index("game_id")
    .loc[saved_game_ids]
    .reset_index()
)

assert np.array_equal(
    future_games["game_id"].astype(str),
    saved_game_ids
)

home_wins = saved["home_wins"]

assert home_wins.shape == (
    N_SIMULATIONS,
    len(future_games)
)

print("Simulated outcomes aligned with future games.")

Simulated outcomes aligned with future games.


In [115]:
division_wins = np.zeros(
    (N_SIMULATIONS, 32), dtype=np.int16
)

division_losses = np.zeros_like(division_wins)
division_ties = np.zeros_like(division_wins)

conference_wins = np.zeros_like(division_wins)
conference_losses = np.zeros_like(division_wins)
conference_ties = np.zeros_like(division_wins)

for game in completed_games.itertuples(index=False):

    h = team_to_idx[game.home_team]
    a = team_to_idx[game.away_team]

    home_win = game.home_score > game.away_score
    away_win = game.away_score > game.home_score
    tie = game.home_score == game.away_score

    if game.is_division_game:
        division_wins[:, h] += int(home_win)
        division_wins[:, a] += int(away_win)
        division_losses[:, h] += int(away_win)
        division_losses[:, a] += int(home_win)
        division_ties[:, h] += int(tie)
        division_ties[:, a] += int(tie)

    if game.is_conference_game:
        conference_wins[:, h] += int(home_win)
        conference_wins[:, a] += int(away_win)
        conference_losses[:, h] += int(away_win)
        conference_losses[:, a] += int(home_win)
        conference_ties[:, h] += int(tie)
        conference_ties[:, a] += int(tie)

print("Actual division and conference records loaded.")

Actual division and conference records loaded.


In [116]:
for i, game in enumerate(
    future_games.itertuples(index=False)
):
    h = team_to_idx[game.home_team]
    a = team_to_idx[game.away_team]

    hw = home_wins[:, i]
    aw = ~hw

    if game.is_division_game:
        division_wins[:, h] += hw
        division_losses[:, h] += aw
        division_wins[:, a] += aw
        division_losses[:, a] += hw

    if game.is_conference_game:
        conference_wins[:, h] += hw
        conference_losses[:, h] += aw
        conference_wins[:, a] += aw
        conference_losses[:, a] += hw

print("Simulated division and conference records calculated.")

Simulated division and conference records calculated.


In [117]:
def winning_percentage(wins, losses, ties):
    total = wins + losses + ties

    return np.divide(
        wins + 0.5 * ties,
        total,
        out=np.zeros_like(wins, dtype=float),
        where=total > 0
    )

overall_pct = winning_percentage(
    simulated_wins,
    simulated_losses,
    simulated_ties
)

division_pct = winning_percentage(
    division_wins,
    division_losses,
    division_ties
)

conference_pct = winning_percentage(
    conference_wins,
    conference_losses,
    conference_ties
)

assert overall_pct.shape == (N_SIMULATIONS, 32)
assert division_pct.shape == (N_SIMULATIONS, 32)
assert conference_pct.shape == (N_SIMULATIONS, 32)

print("Winning percentages calculated.")

Winning percentages calculated.


In [118]:
assert np.all(
    division_wins + division_losses + division_ties <= 6
)

assert np.all(
    conference_wins
    + conference_losses
    + conference_ties <= 12
)

assert np.all(
    division_wins <= conference_wins
)

assert np.all(
    division_losses <= conference_losses
)

np.savez_compressed(
    ROS_OUTPUT_DIR / "tiebreaker_records.npz",
    team_names=np.array(team_names),
    overall_pct=overall_pct,
    division_pct=division_pct,
    conference_pct=conference_pct,
    division_wins=division_wins,
    division_losses=division_losses,
    division_ties=division_ties,
    conference_wins=conference_wins,
    conference_losses=conference_losses,
    conference_ties=conference_ties
)

print("Division and conference records saved.")

Division and conference records saved.


In [119]:
simulation_number = 0

review = team_info.merge(
    pd.DataFrame({
        "team": team_names,
        "wins": simulated_wins[simulation_number],
        "losses": simulated_losses[simulation_number],
        "ties": simulated_ties[simulation_number],
        "overall_pct": overall_pct[simulation_number],
        "division_pct": division_pct[simulation_number],
        "conference_pct": conference_pct[simulation_number]
    }),
    on="team",
    validate="one_to_one"
)

display(
    review.sort_values(
        ["conference", "division", "overall_pct"],
        ascending=[True, True, False]
    ).round(3)
)

,team,division,conference,wins,losses,ties,overall_pct,division_pct,conference_pct
0,BUF,AFC East,AFC,13,4,0,0.765,0.500,0.750
2,NE,AFC East,AFC,9,8,0,0.529,0.667,0.667
1,MIA,AFC East,AFC,7,10,0,0.412,0.667,0.583
3,NYJ,AFC East,AFC,7,10,0,0.412,0.167,0.500
5,CIN,AFC North,AFC,13,4,0,0.765,0.833,0.750
7,PIT,AFC North,AFC,11,6,0,0.647,0.500,0.500
6,CLE,AFC North,AFC,9,8,0,0.529,0.333,0.500
4,BAL,AFC North,AFC,8,9,0,0.471,0.333,0.500
10,JAX,AFC South,AFC,13,4,0,0.765,0.833,0.750
8,HOU,AFC South,AFC,5,12,0,0.294,0.500,0.250


In [120]:
# Check simulated season 0 against stored records

sim_idx = 0

home_win_results = home_wins[sim_idx]

simulated_results = future_games[
    [
        "game_id",
        "week",
        "home_team",
        "away_team",
        "is_division_game",
        "is_conference_game"
    ]
].copy()

simulated_results["winner"] = np.where(
    home_win_results,
    simulated_results["home_team"],
    simulated_results["away_team"]
)

simulated_results["loser"] = np.where(
    home_win_results,
    simulated_results["away_team"],
    simulated_results["home_team"]
)

# Reconstruct wins from completed and simulated games
actual_wins = {
    team: int(wins)
    for team, wins in zip(team_names, current_wins)
}

for winner in simulated_results["winner"]:
    actual_wins[winner] += 1

for team in team_names:
    idx = team_to_idx[team]

    assert actual_wins[team] == simulated_wins[sim_idx, idx], (
        f"Win count mismatch for {team}"
    )

print("Game-level outcomes match simulated final records.")

Game-level outcomes match simulated final records.


In [121]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parents[1]

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rest_of_season.src.nfl_tiebreakers import (
    winning_percentage,
    head_to_head_record
)

# Test NFL winning percentage
assert winning_percentage(10, 7, 0) == 10 / 17
assert winning_percentage(9, 7, 1) == 9.5 / 17
assert winning_percentage(0, 0, 0) == 0.0

# Construct a simple hypothetical head-to-head matchup
test_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": None}
]

result = head_to_head_record(
    "BAL",
    {"PIT", "CIN"},
    test_games
)

assert result["wins"] == 2
assert result["losses"] == 1
assert result["ties"] == 1
assert result["pct"] == 0.625

print("All initial NFL tiebreaker tests passed.")
print(result)

All initial NFL tiebreaker tests passed.
{'wins': 2, 'losses': 1, 'ties': 1, 'pct': 0.625}


In [122]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    winning_percentage,
    head_to_head_record,
    build_season_results
)

print("Tiebreaker module reloaded successfully.")

Tiebreaker module reloaded successfully.


In [123]:
from rest_of_season.src.nfl_tiebreakers import build_season_results

# Build the full season from simulation #0
season_results = build_season_results(
    completed_games,
    future_games,
    home_wins,
    sim_idx=0
)

assert len(season_results) == 272

# Every game should have a valid winner or a tie
for game in season_results:
    assert game["winner"] in {
        game["home_team"],
        game["away_team"],
        None
    }

# Check Baltimore's record against AFC North opponents
bal_division = head_to_head_record(
    "BAL",
    {"PIT", "CIN", "CLE"},
    season_results
)

bal_idx = team_to_idx["BAL"]

assert bal_division["wins"] == division_wins[0, bal_idx]
assert bal_division["losses"] == division_losses[0, bal_idx]
assert bal_division["ties"] == division_ties[0, bal_idx]

print("All 272 games reconstructed successfully.")
print("Baltimore division record:", bal_division)

All 272 games reconstructed successfully.
Baltimore division record: {'wins': 2, 'losses': 4, 'ties': 0, 'pct': 0.3333333333333333}


In [124]:

from rest_of_season.src.nfl_tiebreakers import (
    two_team_division_tiebreaker
)

# Test a two-team division tiebreaker using
# hypothetical Cincinnati and Pittsburgh records.

test_games = [
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"}
]

test_division_records = {
    "CIN": {"wins": 5, "losses": 1, "ties": 0},
    "PIT": {"wins": 4, "losses": 2, "ties": 0}
}

test_conference_records = {
    "CIN": {"wins": 8, "losses": 4, "ties": 0},
    "PIT": {"wins": 7, "losses": 5, "ties": 0}
}

winner, criterion = two_team_division_tiebreaker(
    "CIN",
    "PIT",
    test_games,
    test_division_records,
    test_conference_records
)

assert winner == "CIN"
assert criterion == "division_record"

print("Two-team division tiebreaker test passed.")
print("Winner:", winner)
print("Deciding criterion:", criterion)


Two-team division tiebreaker test passed.
Winner: CIN
Deciding criterion: division_record


In [125]:
# Test Cincinnati vs. Pittsburgh using simulation #0

division_records = {}
conference_records = {}

for team in ["CIN", "PIT"]:
    idx = team_to_idx[team]

    division_records[team] = {
        "wins": int(division_wins[0, idx]),
        "losses": int(division_losses[0, idx]),
        "ties": int(division_ties[0, idx])
    }

    conference_records[team] = {
        "wins": int(conference_wins[0, idx]),
        "losses": int(conference_losses[0, idx]),
        "ties": int(conference_ties[0, idx])
    }

winner, criterion = two_team_division_tiebreaker(
    "CIN",
    "PIT",
    season_results,
    division_records,
    conference_records
)

assert winner == "CIN"
assert criterion == "division_record"

print("Simulation #0 AFC North tie:")
print("Cincinnati:", division_records["CIN"])
print("Pittsburgh:", division_records["PIT"])
print("Winner:", winner)
print("Deciding criterion:", criterion)

Simulation #0 AFC North tie:
Cincinnati: {'wins': 5, 'losses': 1, 'ties': 0}
Pittsburgh: {'wins': 3, 'losses': 3, 'ties': 0}
Winner: CIN
Deciding criterion: division_record


In [126]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    common_opponents,
    common_games_record
)

# Use our reconstructed simulation #0
shared = common_opponents(
    "CIN",
    "PIT",
    season_results
)

cin_common = common_games_record(
    "CIN",
    shared,
    season_results
)

pit_common = common_games_record(
    "PIT",
    shared,
    season_results
)

print("CIN-PIT common opponents:", sorted(shared))
print("Cincinnati common-games record:", cin_common)
print("Pittsburgh common-games record:", pit_common)

assert len(shared) > 0

print("Common-games calculations passed.")

CIN-PIT common opponents: ['ATL', 'BAL', 'CAR', 'CLE', 'HOU', 'IND', 'JAX', 'NO', 'TB', 'TEN']
Cincinnati common-games record: {'wins': 10, 'losses': 2, 'ties': 0, 'pct': 0.8333333333333334}
Pittsburgh common-games record: {'wins': 9, 'losses': 3, 'ties': 0, 'pct': 0.75}
Common-games calculations passed.


In [127]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    two_team_division_tiebreaker
)

In [128]:
cin_idx = team_to_idx["CIN"]
pit_idx = team_to_idx["PIT"]

division_records = {}
conference_records = {}

for team in ["CIN", "PIT"]:
    idx = team_to_idx[team]

    division_records[team] = {
        "wins": int(division_wins[0, idx]),
        "losses": int(division_losses[0, idx]),
        "ties": int(division_ties[0, idx])
    }

    conference_records[team] = {
        "wins": int(conference_wins[0, idx]),
        "losses": int(conference_losses[0, idx]),
        "ties": int(conference_ties[0, idx])
    }

winner, criterion = two_team_division_tiebreaker(
    "CIN",
    "PIT",
    season_results,
    division_records,
    conference_records
)

assert winner == "CIN"
assert criterion == "division_record"

print("Four-criterion tiebreaker test passed.")
print("Winner:", winner)
print("Deciding criterion:", criterion)

Four-criterion tiebreaker test passed.
Winner: CIN
Deciding criterion: division_record


In [129]:
test_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"}
]

test_division = {
    "BAL": {"wins": 3, "losses": 3, "ties": 0},
    "PIT": {"wins": 3, "losses": 3, "ties": 0}
}

test_conference = {
    "BAL": {"wins": 9, "losses": 3, "ties": 0},
    "PIT": {"wins": 8, "losses": 4, "ties": 0}
}

winner, criterion = two_team_division_tiebreaker(
    "BAL",
    "PIT",
    test_games,
    test_division,
    test_conference
)

assert winner == "BAL"
assert criterion == "conference_record"

print("Conference-record tiebreaker test passed.")
print("Winner:", winner)
print("Deciding criterion:", criterion)

Conference-record tiebreaker test passed.
Winner: BAL
Deciding criterion: conference_record


In [130]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    calculate_final_records,
    strength_of_record
)

final_records = calculate_final_records(season_results)

# Verify reconstructed records match the simulation
for team in team_names:
    idx = team_to_idx[team]

    assert final_records[team]["wins"] == simulated_wins[0, idx]
    assert final_records[team]["losses"] == simulated_losses[0, idx]
    assert final_records[team]["ties"] == simulated_ties[0, idx]

print("Final records verified against Monte Carlo results.")

for team in ["BAL", "CIN", "PIT", "CLE"]:
    sov = strength_of_record(
        team,
        season_results,
        final_records,
        victories_only=True
    )

    sos = strength_of_record(
        team,
        season_results,
        final_records,
        victories_only=False
    )

    print(f"{team}: SOV={sov:.4f}, SOS={sos:.4f}")

Final records verified against Monte Carlo results.
BAL: SOV=0.4485, SOS=0.5052
CIN: SOV=0.4231, SOS=0.4723
PIT: SOV=0.4064, SOS=0.4740
CLE: SOV=0.4346, SOS=0.4965


In [131]:
for team in team_names:
    sov = strength_of_record(
        team, season_results, final_records, victories_only=True
    )

    sos = strength_of_record(
        team, season_results, final_records, victories_only=False
    )

    assert 0 <= sov <= 1
    assert 0 <= sos <= 1

print("All 32 teams have valid SOV and SOS values.")

All 32 teams have valid SOV and SOS values.


In [132]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    two_team_division_tiebreaker
)

# Reconstruct simulation #0's division and conference records
division_records = {}
conference_records = {}

for team in team_names:
    idx = team_to_idx[team]

    division_records[team] = {
        "wins": int(division_wins[0, idx]),
        "losses": int(division_losses[0, idx]),
        "ties": int(division_ties[0, idx])
    }

    conference_records[team] = {
        "wins": int(conference_wins[0, idx]),
        "losses": int(conference_losses[0, idx]),
        "ties": int(conference_ties[0, idx])
    }

# Cincinnati and Pittsburgh remain tied at 10-7
winner, criterion = two_team_division_tiebreaker(
    "CIN",
    "PIT",
    season_results,
    division_records,
    conference_records
)

assert winner == "CIN"
assert criterion == "division_record"

print("Six-criterion tiebreaker test passed.")
print("Winner:", winner)
print("Deciding criterion:", criterion)

Six-criterion tiebreaker test passed.
Winner: CIN
Deciding criterion: division_record


In [133]:
from rest_of_season.src.nfl_tiebreakers import (
    strength_of_record,
    calculate_final_records
)

test_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "CLE", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "CLE", "winner": "CIN"},
    {"home_team": "CLE", "away_team": "BAL", "winner": "CLE"}
]

test_records = calculate_final_records(test_games)

bal_sov = strength_of_record(
    "BAL", test_games, test_records, victories_only=True
)

pit_sov = strength_of_record(
    "PIT", test_games, test_records, victories_only=True
)

assert bal_sov > pit_sov

print("Strength-of-victory calculation test passed.")
print(f"BAL SOV: {bal_sov:.4f}")
print(f"PIT SOV: {pit_sov:.4f}")

Strength-of-victory calculation test passed.
BAL SOV: 0.5000
PIT SOV: 0.3333


In [134]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    multi_team_head_to_head_records
)

test_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"}
]

teams = ["BAL", "PIT", "CIN"]

records = multi_team_head_to_head_records(
    teams,
    test_games
)

for team, record in records.items():
    print(team, record)

assert records["BAL"]["wins"] == 3
assert records["BAL"]["losses"] == 1

assert records["PIT"]["wins"] == 2
assert records["PIT"]["losses"] == 2

assert records["CIN"]["wins"] == 1
assert records["CIN"]["losses"] == 3

print("Multi-team head-to-head test passed.")

BAL {'wins': 3, 'losses': 1, 'ties': 0, 'pct': 0.75}
PIT {'wins': 2, 'losses': 2, 'ties': 0, 'pct': 0.5}
CIN {'wins': 1, 'losses': 3, 'ties': 0, 'pct': 0.25}
Multi-team head-to-head test passed.


In [135]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    multi_team_division_tiebreaker
)

test_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"}
]

test_division = {
    team: {"wins": 3, "losses": 3, "ties": 0}
    for team in ["BAL", "PIT", "CIN"]
}

test_conference = {
    team: {"wins": 8, "losses": 4, "ties": 0}
    for team in ["BAL", "PIT", "CIN"]
}

winner, criterion = multi_team_division_tiebreaker(
    ["BAL", "PIT", "CIN"],
    test_games,
    test_division,
    test_conference
)
print("Actual winner:", winner)
print("Actual criterion:", criterion)

assert winner == "BAL"
assert criterion == "multi_team_head_to_head"

print("Multi-team outright winner test passed.")
print("Winner:", winner)
print("Criterion:", criterion)

Actual winner: BAL
Actual criterion: multi_team_head_to_head
Multi-team outright winner test passed.
Winner: BAL
Criterion: multi_team_head_to_head


In [136]:
restart_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "PIT"}
]

restart_division = {
    "BAL": {"wins": 4, "losses": 2, "ties": 0},
    "PIT": {"wins": 3, "losses": 3, "ties": 0},
    "CIN": {"wins": 1, "losses": 5, "ties": 0}
}

restart_conference = {
    team: {"wins": 8, "losses": 4, "ties": 0}
    for team in ["BAL", "PIT", "CIN"]
}

winner, criterion = multi_team_division_tiebreaker(
    ["BAL", "PIT", "CIN"],
    restart_games,
    restart_division,
    restart_conference
)

assert winner == "BAL"
assert criterion == "restart_two_team_division_record"

print("Two-team restart test passed.")
print("Winner:", winner)
print("Criterion:", criterion)

Two-team restart test passed.
Winner: BAL
Criterion: restart_two_team_division_record


In [137]:
test_games = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "CIN"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"}
]

test_division = {
    "BAL": {"wins": 5, "losses": 1, "ties": 0},
    "PIT": {"wins": 4, "losses": 2, "ties": 0},
    "CIN": {"wins": 3, "losses": 3, "ties": 0}
}

test_conference = {
    team: {"wins": 8, "losses": 4, "ties": 0}
    for team in ["BAL", "PIT", "CIN"]
}

winner, criterion = multi_team_division_tiebreaker(
    ["BAL", "PIT", "CIN"],
    test_games,
    test_division,
    test_conference
)

assert winner == "BAL"
assert criterion == "multi_team_division_record"

print("Multi-team division-record test passed.")
print("Winner:", winner)
print("Criterion:", criterion)

Multi-team division-record test passed.
Winner: BAL
Criterion: multi_team_division_record


In [138]:
# Four-team head-to-head records:
# BAL: 4-2
# PIT: 4-2
# CIN: 4-2
# CLE: 0-6

test_games = []

for opponent in ["BAL", "PIT", "CIN"]:
    test_games.append({
        "home_team": opponent,
        "away_team": "CLE",
        "winner": opponent
    })
    test_games.append({
        "home_team": "CLE",
        "away_team": opponent,
        "winner": opponent
    })

# BAL, PIT, CIN split their head-to-head games
for team_a, team_b in [
    ("BAL", "PIT"),
    ("BAL", "CIN"),
    ("PIT", "CIN")
]:
    test_games.append({
        "home_team": team_a,
        "away_team": team_b,
        "winner": team_a
    })
    test_games.append({
        "home_team": team_b,
        "away_team": team_a,
        "winner": team_b
    })

test_division = {
    "BAL": {"wins": 5, "losses": 1, "ties": 0},
    "PIT": {"wins": 4, "losses": 2, "ties": 0},
    "CIN": {"wins": 3, "losses": 3, "ties": 0},
    "CLE": {"wins": 0, "losses": 6, "ties": 0}
}

test_conference = {
    team: {"wins": 8, "losses": 4, "ties": 0}
    for team in ["BAL", "PIT", "CIN", "CLE"]
}

records = multi_team_head_to_head_records(
    ["BAL", "PIT", "CIN", "CLE"],
    test_games
)

for team, record in records.items():
    print(team, record)

winner, criterion = multi_team_division_tiebreaker(
    ["BAL", "PIT", "CIN", "CLE"],
    test_games,
    test_division,
    test_conference
)

print(f"Actual winner: {winner}")
print(f"Actual criterion: {criterion!r}")

print("\nFour-team reduction test passed.")
print("Winner:", winner)
print("Criterion:", criterion)

BAL {'wins': 4, 'losses': 2, 'ties': 0, 'pct': 0.6666666666666666}
PIT {'wins': 4, 'losses': 2, 'ties': 0, 'pct': 0.6666666666666666}
CIN {'wins': 4, 'losses': 2, 'ties': 0, 'pct': 0.6666666666666666}
CLE {'wins': 0, 'losses': 6, 'ties': 0, 'pct': 0.0}
Actual winner: BAL
Actual criterion: 'restart_three_team_multi_team_division_record'

Four-team reduction test passed.
Winner: BAL
Criterion: restart_three_team_multi_team_division_record


In [139]:
restart_games = [
    # BAL vs PIT: Pittsburgh wins both
    {"home_team": "BAL", "away_team": "PIT", "winner": "PIT"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},

    # BAL vs CIN: Baltimore wins both
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "BAL"},

    # PIT vs CIN: Cincinnati wins both
    {"home_team": "PIT", "away_team": "CIN", "winner": "CIN"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"}
]

restart_division = {
    "BAL": {"wins": 4, "losses": 2, "ties": 0},
    "PIT": {"wins": 4, "losses": 2, "ties": 0},
    "CIN": {"wins": 3, "losses": 3, "ties": 0}
}

restart_conference = {
    team: {"wins": 8, "losses": 4, "ties": 0}
    for team in ["BAL", "PIT", "CIN"]
}

h2h = multi_team_head_to_head_records(
    ["BAL", "PIT", "CIN"],
    restart_games
)

for team, record in h2h.items():
    print(team, record)

winner, criterion = multi_team_division_tiebreaker(
    ["BAL", "PIT", "CIN"],
    restart_games,
    restart_division,
    restart_conference
)

assert winner == "PIT"
assert criterion == "restart_two_team_head_to_head"

print("\nDivision-record reduction test passed.")
print("Winner:", winner)
print("Criterion:", criterion)

BAL {'wins': 2, 'losses': 2, 'ties': 0, 'pct': 0.5}
PIT {'wins': 2, 'losses': 2, 'ties': 0, 'pct': 0.5}
CIN {'wins': 2, 'losses': 2, 'ties': 0, 'pct': 0.5}

Division-record reduction test passed.
Winner: PIT
Criterion: restart_two_team_head_to_head


In [140]:
from rest_of_season.src.nfl_tiebreakers import (
    calculate_final_records,
    winning_percentage
)

# Validate 100 simulations before scaling up
n_test = min(100, N_SIMULATIONS)

for sim_idx in range(n_test):

    results = nfl_tiebreakers.build_season_results(
        completed_games,
        future_games,
        home_wins,
        sim_idx
    )

    records = calculate_final_records(results)

    for team in team_names:
        idx = team_to_idx[team]

        assert records[team]["wins"] == simulated_wins[sim_idx, idx]
        assert records[team]["losses"] == simulated_losses[sim_idx, idx]
        assert records[team]["ties"] == simulated_ties[sim_idx, idx]

        total_games = (
            records[team]["wins"]
            + records[team]["losses"]
            + records[team]["ties"]
        )

        assert total_games == 17, (
            f"Simulation {sim_idx}, {team}: "
            f"expected 17 games, found {total_games}"
        )

print(f"All {n_test} simulated seasons passed record validation.")

All 100 simulated seasons passed record validation.


In [141]:
for sim_idx in range(n_test):

    for team in team_names:
        idx = team_to_idx[team]

        div_games = (
            division_wins[sim_idx, idx]
            + division_losses[sim_idx, idx]
            + division_ties[sim_idx, idx]
        )

        conf_games = (
            conference_wins[sim_idx, idx]
            + conference_losses[sim_idx, idx]
            + conference_ties[sim_idx, idx]
        )

        assert div_games == 6, (
            f"Simulation {sim_idx}, {team}: "
            f"expected 6 division games, found {div_games}"
        )

        assert 0 <= conf_games <= 17

print(f"All {n_test} simulated seasons passed division/conference validation.")

All 100 simulated seasons passed division/conference validation.


In [142]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

from rest_of_season.src.nfl_tiebreakers import (
    select_division_winner,
    calculate_final_records
)

# Reconstruct simulation #0
sim_idx = 0

results = nfl_tiebreakers.build_season_results(
    completed_games,
    future_games,
    home_wins,
    sim_idx
)

final_records = calculate_final_records(results)

division_records = {}
conference_records = {}

for team in team_names:
    idx = team_to_idx[team]

    division_records[team] = {
        "wins": int(division_wins[sim_idx, idx]),
        "losses": int(division_losses[sim_idx, idx]),
        "ties": int(division_ties[sim_idx, idx])
    }

    conference_records[team] = {
        "wins": int(conference_wins[sim_idx, idx]),
        "losses": int(conference_losses[sim_idx, idx]),
        "ties": int(conference_ties[sim_idx, idx])
    }

division_winners = {}

for division, teams in DIVISIONS.items():

    winner, criterion = select_division_winner(
        teams,
        final_records,
        results,
        division_records,
        conference_records
    )

    division_winners[division] = winner

    print(f"{division}: {winner} ({criterion})")

assert len(division_winners) == 8
assert all(winner is not None for winner in division_winners.values())
assert len(set(division_winners.values())) == 8

print("\nSimulation #0 division-winner selection passed.")

AFC East: BUF (overall_record)
AFC North: CIN (overall_record)
AFC South: JAX (overall_record)
AFC West: KC (overall_record)
NFC East: DAL (overall_record)
NFC North: MIN (overall_record)
NFC South: TB (overall_record)
NFC West: SEA (overall_record)

Simulation #0 division-winner selection passed.


In [143]:
from collections import Counter
import time

n_test = 100

criterion_counts = Counter()
unresolved = []
division_win_counts = Counter()

start = time.perf_counter()

for sim_idx in range(n_test):

    results = nfl_tiebreakers.build_season_results(
        completed_games,
        future_games,
        home_wins,
        sim_idx
    )

    final_records = nfl_tiebreakers.calculate_final_records(results)

    div_records = {}
    conf_records = {}

    for team in team_names:
        idx = team_to_idx[team]

        div_records[team] = {
            "wins": int(division_wins[sim_idx, idx]),
            "losses": int(division_losses[sim_idx, idx]),
            "ties": int(division_ties[sim_idx, idx])
        }

        conf_records[team] = {
            "wins": int(conference_wins[sim_idx, idx]),
            "losses": int(conference_losses[sim_idx, idx]),
            "ties": int(conference_ties[sim_idx, idx])
        }

    for division, teams in DIVISIONS.items():

        winner, criterion = nfl_tiebreakers.select_division_winner(
            teams,
            final_records,
            results,
            div_records,
            conf_records
        )

        criterion_counts[criterion] += 1

        if winner is None:
            unresolved.append((sim_idx, division, criterion))
            continue

        assert winner in teams

        best_pct = max(
            nfl_tiebreakers.winning_percentage(
                final_records[t]["wins"],
                final_records[t]["losses"],
                final_records[t]["ties"]
            )
            for t in teams
        )

        winner_record = final_records[winner]

        winner_pct = nfl_tiebreakers.winning_percentage(
            winner_record["wins"],
            winner_record["losses"],
            winner_record["ties"]
        )

        assert winner_pct == best_pct, (
            f"Simulation {sim_idx}: {winner} won {division} "
            "without the best overall record"
        )

        division_win_counts[winner] += 1

elapsed = time.perf_counter() - start

print(f"Simulations tested: {n_test}")
print(f"Division selections checked: {n_test * 8}")
print(f"Unresolved selections: {len(unresolved)}")
print(f"Elapsed time: {elapsed:.2f} seconds")

print("\nDeciding criteria:")
for criterion, count in criterion_counts.most_common():
    print(f"  {criterion}: {count}")

print("\nUnresolved examples:")
print(unresolved[:10])

Simulations tested: 100
Division selections checked: 800
Unresolved selections: 0
Elapsed time: 1.05 seconds

Deciding criteria:
  overall_record: 690
  head_to_head: 42
  division_record: 37
  multi_team_head_to_head: 10
  common_games: 8
  strength_of_victory: 4
  restart_two_team_common_games: 3
  multi_team_division_record: 3
  conference_record: 2
  strength_of_schedule: 1

Unresolved examples:
[]


In [144]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

<module 'rest_of_season.src.nfl_tiebreakers' from 'C:\\Users\\efriedman\\Desktop\\NFL-Season-Projections\\rest_of_season\\src\\nfl_tiebreakers.py'>

In [145]:
sim_idx = 0

results = nfl_tiebreakers.build_season_results(
    completed_games,
    future_games,
    home_wins,
    sim_idx
)

final_records = nfl_tiebreakers.calculate_final_records(
    results
)

div_records = {}
conf_records = {}

for team in team_names:
    idx = team_to_idx[team]

    div_records[team] = {
        "wins": int(division_wins[sim_idx, idx]),
        "losses": int(division_losses[sim_idx, idx]),
        "ties": int(division_ties[sim_idx, idx])
    }

    conf_records[team] = {
        "wins": int(conference_wins[sim_idx, idx]),
        "losses": int(conference_losses[sim_idx, idx]),
        "ties": int(conference_ties[sim_idx, idx])
    }

print("Projected division standings — Simulation 0\n")

for division, teams in DIVISIONS.items():

    ranking = rank_division_teams(
        teams,
        final_records,
        results,
        div_records,
        conf_records
    )

    print(division)

    if ranking is None:
        print("  Unresolved division ranking\n")
        continue

    for position, team in enumerate(ranking, start=1):
        record = final_records[team]

        print(
            f"  {position}. {team} "
            f"({record['wins']}-{record['losses']}-{record['ties']})"
        )

    print()

Projected division standings — Simulation 0

AFC East
  1. BUF (13-4-0)
  2. NE (9-8-0)
  3. MIA (7-10-0)
  4. NYJ (7-10-0)

AFC North
  1. CIN (13-4-0)
  2. PIT (11-6-0)
  3. CLE (9-8-0)
  4. BAL (8-9-0)

AFC South
  1. JAX (13-4-0)
  2. TEN (5-12-0)
  3. HOU (5-12-0)
  4. IND (2-15-0)

AFC West
  1. KC (13-4-0)
  2. DEN (9-8-0)
  3. LV (8-9-0)
  4. LAC (2-15-0)

NFC East
  1. DAL (10-7-0)
  2. NYG (9-7-1)
  3. WAS (5-11-1)
  4. PHI (4-13-0)

NFC North
  1. MIN (13-4-0)
  2. DET (12-5-0)
  3. CHI (10-7-0)
  4. GB (9-8-0)

NFC South
  1. TB (10-7-0)
  2. CAR (8-9-0)
  3. ATL (6-11-0)
  4. NO (6-11-0)

NFC West
  1. SEA (11-6-0)
  2. SF (9-8-0)
  3. LA (9-8-0)
  4. ARI (6-11-0)



In [146]:
from collections import Counter
import pandas as pd
import time

division_win_counts = Counter()
criterion_counts = Counter()
unresolved = []

start = time.perf_counter()

for sim_idx in range(N_SIMULATIONS):

    results = nfl_tiebreakers.build_season_results(
        completed_games,
        future_games,
        home_wins,
        sim_idx
    )

    final_records = nfl_tiebreakers.calculate_final_records(results)

    div_records = {}
    conf_records = {}

    for team in team_names:
        idx = team_to_idx[team]

        div_records[team] = {
            "wins": int(division_wins[sim_idx, idx]),
            "losses": int(division_losses[sim_idx, idx]),
            "ties": int(division_ties[sim_idx, idx])
        }

        conf_records[team] = {
            "wins": int(conference_wins[sim_idx, idx]),
            "losses": int(conference_losses[sim_idx, idx]),
            "ties": int(conference_ties[sim_idx, idx])
        }

    for division, teams in DIVISIONS.items():

        winner, criterion = nfl_tiebreakers.select_division_winner(
            teams,
            final_records,
            results,
            div_records,
            conf_records
        )

        criterion_counts[criterion] += 1

        if winner is None:
            unresolved.append({
                "simulation": sim_idx,
                "division": division,
                "criterion": criterion
            })
            continue

        division_win_counts[winner] += 1

    if (sim_idx + 1) % 10000 == 0:
        print(f"Completed {sim_idx + 1:,} simulations")

elapsed = time.perf_counter() - start

print(f"\nCompleted in {elapsed:.2f} seconds")
print(f"Unresolved division selections: {len(unresolved)}")

division_probabilities = pd.DataFrame([
    {
        "team": team,
        "division": division,
        "division_titles": division_win_counts[team],
        "division_title_probability": (
            division_win_counts[team] / N_SIMULATIONS
        )
    }
    for division, teams in DIVISIONS.items()
    for team in teams
])

division_probabilities = division_probabilities.sort_values(
    ["division", "division_title_probability"],
    ascending=[True, False]
).reset_index(drop=True)

division_probabilities.to_csv(
    ROS_OUTPUT_DIR / "division_title_probabilities_v2.csv",
    index=False
)

division_probabilities.to_parquet(
    ROS_OUTPUT_DIR / "division_title_probabilities_v2.parquet",
    index=False
)

pd.DataFrame(unresolved).to_csv(
    ROS_OUTPUT_DIR / "unresolved_division_ties_v2.csv",
    index=False
)

print("\nDivision-title probabilities:")
print(division_probabilities.to_string(index=False))

print("\nTiebreaker usage:")
for criterion, count in criterion_counts.most_common():
    print(f"{criterion}: {count}")

Completed 10,000 simulations
Completed 20,000 simulations
Completed 30,000 simulations
Completed 40,000 simulations
Completed 50,000 simulations

Completed in 514.79 seconds
Unresolved division selections: 0

Division-title probabilities:
team  division  division_titles  division_title_probability
 BUF  AFC East            42013                     0.84026
  NE  AFC East             7974                     0.15948
 NYJ  AFC East                7                     0.00014
 MIA  AFC East                6                     0.00012
 PIT AFC North            26040                     0.52080
 CIN AFC North            19621                     0.39242
 CLE AFC North             2628                     0.05256
 BAL AFC North             1711                     0.03422
 JAX AFC South            46147                     0.92294
 HOU AFC South             3312                     0.06624
 IND AFC South              519                     0.01038
 TEN AFC South               22          

In [148]:
unresolved_df = pd.DataFrame(unresolved)

print("Unresolved division tiebreakers:")
print(unresolved_df.to_string(index=False))

print("\nUnresolved by division:")

if unresolved_df.empty:
    print("No unresolved division tiebreakers.")
else:
    print(unresolved_df["division"].value_counts())

Unresolved division tiebreakers:
Empty DataFrame
Columns: []
Index: []

Unresolved by division:
No unresolved division tiebreakers.


In [149]:
for case in unresolved:
    sim_idx = case["simulation"]
    division = case["division"]

    teams = DIVISIONS[division]

    records = []

    for team in teams:
        idx = team_to_idx[team]

        wins = int(simulated_wins[sim_idx, idx])
        losses = int(simulated_losses[sim_idx, idx])
        ties = int(simulated_ties[sim_idx, idx])

        records.append({
            "team": team,
            "wins": wins,
            "losses": losses,
            "ties": ties,
            "division_wins": int(division_wins[sim_idx, idx]),
            "division_losses": int(division_losses[sim_idx, idx]),
            "conference_wins": int(conference_wins[sim_idx, idx]),
            "conference_losses": int(conference_losses[sim_idx, idx])
        })

    print(f"\nSimulation {sim_idx} — {division}")
    print(pd.DataFrame(records).to_string(index=False))

In [150]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

print("Tiebreaker module reloaded.")

Tiebreaker module reloaded.


In [151]:
# Synthetic four-team tie:
# BAL, PIT and CIN have identical head-to-head records.
# CLE loses every game against the other three teams.
#
# Once CLE is eliminated, the three-team process should restart.

test_teams = ["BAL", "PIT", "CIN", "CLE"]

test_results = []

# Each of BAL, PIT and CIN splits with the other two
for a, b in [
    ("BAL", "PIT"),
    ("BAL", "CIN"),
    ("PIT", "CIN")
]:
    test_results.append({
        "home_team": a,
        "away_team": b,
        "winner": a
    })
    test_results.append({
        "home_team": b,
        "away_team": a,
        "winner": b
    })

# All three teams sweep CLE
for team in ["BAL", "PIT", "CIN"]:
    for _ in range(2):
        test_results.append({
            "home_team": team,
            "away_team": "CLE",
            "winner": team
        })

test_division_records = {
    "BAL": {"wins": 5, "losses": 1, "ties": 0},
    "PIT": {"wins": 4, "losses": 2, "ties": 0},
    "CIN": {"wins": 3, "losses": 3, "ties": 0},
    "CLE": {"wins": 0, "losses": 6, "ties": 0}
}

test_conference_records = {
    team: {"wins": 8, "losses": 4, "ties": 0}
    for team in test_teams
}

winner, criterion = nfl_tiebreakers.multi_team_division_tiebreaker(
    test_teams,
    test_results,
    test_division_records,
    test_conference_records
)

print("Winner:", winner)
print("Criterion:", criterion)

assert winner == "BAL"
assert criterion == "restart_three_team_multi_team_division_record"

print("Four-to-three restart test passed.")

Winner: BAL
Criterion: restart_three_team_multi_team_division_record
Four-to-three restart test passed.


In [152]:
# Check common opponents for two AFC North teams
test_team_a = "BAL"
test_team_b = "PIT"

shared = nfl_tiebreakers.common_opponents(
    test_team_a,
    test_team_b,
    season_results
)

record_a = nfl_tiebreakers.common_games_record(
    test_team_a,
    shared,
    season_results
)

record_b = nfl_tiebreakers.common_games_record(
    test_team_b,
    shared,
    season_results
)

print("Common opponents:", sorted(shared))

print(f"\n{test_team_a} common-games record:")
print(record_a)

print(f"\n{test_team_b} common-games record:")
print(record_b)

# Validate that neither tied team appears in the
# common-opponent set
assert test_team_a not in shared
assert test_team_b not in shared

# Validate winning percentages
assert 0 <= record_a["pct"] <= 1
assert 0 <= record_b["pct"] <= 1

# NFL division common-games tiebreaker requires
# at least four common games.
games_a = (
    record_a["wins"]
    + record_a["losses"]
    + record_a["ties"]
)

games_b = (
    record_b["wins"]
    + record_b["losses"]
    + record_b["ties"]
)

assert games_a >= 4
assert games_b >= 4

print("\nCommon-games validation passed.")

Common opponents: ['ATL', 'CAR', 'CIN', 'CLE', 'HOU', 'IND', 'JAX', 'NO', 'TB', 'TEN']

BAL common-games record:
{'wins': 6, 'losses': 6, 'ties': 0, 'pct': 0.5}

PIT common-games record:
{'wins': 9, 'losses': 3, 'ties': 0, 'pct': 0.75}

Common-games validation passed.


In [153]:
test_groups = [
    ["BAL", "PIT", "CIN"],
    ["BAL", "PIT", "CIN", "CLE"]
]

for teams in test_groups:

    opponent_sets = []

    for team in teams:
        opponents = set()

        for game in season_results:
            if game["home_team"] == team:
                opponents.add(game["away_team"])
            elif game["away_team"] == team:
                opponents.add(game["home_team"])

        opponents -= set(teams)
        opponent_sets.append(opponents)

    shared = set.intersection(*opponent_sets)

    print(f"\nTied teams: {teams}")
    print(f"Common opponents: {sorted(shared)}")

    for team in teams:
        record = nfl_tiebreakers.common_games_record(
            team,
            shared,
            season_results
        )

        games = (
            record["wins"]
            + record["losses"]
            + record["ties"]
        )

        print(
            f"{team}: "
            f"{record['wins']}-{record['losses']}-{record['ties']} "
            f"({games} games, {record['pct']:.3f})"
        )


Tied teams: ['BAL', 'PIT', 'CIN']
Common opponents: ['ATL', 'CAR', 'CLE', 'HOU', 'IND', 'JAX', 'NO', 'TB', 'TEN']
BAL: 6-4-0 (10 games, 0.600)
PIT: 8-2-0 (10 games, 0.800)
CIN: 8-2-0 (10 games, 0.800)

Tied teams: ['BAL', 'PIT', 'CIN', 'CLE']
Common opponents: ['ATL', 'CAR', 'HOU', 'IND', 'JAX', 'NO', 'TB', 'TEN']
BAL: 5-3-0 (8 games, 0.625)
PIT: 7-1-0 (8 games, 0.875)
CIN: 6-2-0 (8 games, 0.750)
CLE: 5-3-0 (8 games, 0.625)


In [154]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

print("Tiebreaker module updated successfully.")

Tiebreaker module updated successfully.


In [155]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

# Scenario 1: BAL sweeps PIT and CIN
test_games_1 = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "BAL"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"},
]

result_1 = nfl_tiebreakers.multi_team_head_to_head_sweep(
    ["BAL", "PIT", "CIN"],
    test_games_1
)

print("Scenario 1:", result_1)
assert result_1 == ("winner", "BAL")


# Scenario 2: CLE loses to every other tied team
test_games_2 = [
    {"home_team": "BAL", "away_team": "PIT", "winner": "BAL"},
    {"home_team": "PIT", "away_team": "BAL", "winner": "PIT"},
    {"home_team": "BAL", "away_team": "CIN", "winner": "BAL"},
    {"home_team": "CIN", "away_team": "BAL", "winner": "CIN"},
    {"home_team": "PIT", "away_team": "CIN", "winner": "PIT"},
    {"home_team": "CIN", "away_team": "PIT", "winner": "CIN"},
]

for team in ["BAL", "PIT", "CIN"]:
    test_games_2.extend([
        {"home_team": team, "away_team": "CLE", "winner": team},
        {"home_team": "CLE", "away_team": team, "winner": team},
    ])

result_2 = nfl_tiebreakers.multi_team_head_to_head_sweep(
    ["BAL", "PIT", "CIN", "CLE"],
    test_games_2
)

print("Scenario 2:", result_2)
assert result_2 == ("eliminate", "CLE")

print("\nBoth head-to-head sweep tests passed.")

Scenario 1: ('winner', 'BAL')
Scenario 2: ('eliminate', 'CLE')

Both head-to-head sweep tests passed.


In [156]:
import importlib
import rest_of_season.src.nfl_tiebreakers as nfl_tiebreakers

importlib.reload(nfl_tiebreakers)

winner, criterion = nfl_tiebreakers.multi_team_division_tiebreaker(
    test_teams,
    test_results,
    test_division_records,
    test_conference_records
)

print("Winner:", winner)
print("Criterion:", criterion)

assert winner == "BAL"
print("Updated multi-team tiebreaker test passed.")

Winner: BAL
Criterion: restart_three_team_multi_team_division_record
Updated multi-team tiebreaker test passed.


In [157]:

# Recheck unresolved division tiebreakers
# from the current week's simulations.

current_unresolved = [
    (case["simulation"], case["division"])
    for case in unresolved
]

print(
    f"Rechecking {len(current_unresolved)} unresolved "
    f"division tiebreakers for Week {TARGET_WEEK}."
)

for sim_idx, division in current_unresolved:

    results = nfl_tiebreakers.build_season_results(
        completed_games,
        future_games,
        home_wins,
        sim_idx
    )

    final_records = nfl_tiebreakers.calculate_final_records(
        results
    )

    div_records = {}
    conf_records = {}

    for team in team_names:
        idx = team_to_idx[team]

        div_records[team] = {
            "wins": int(division_wins[sim_idx, idx]),
            "losses": int(division_losses[sim_idx, idx]),
            "ties": int(division_ties[sim_idx, idx])
        }

        conf_records[team] = {
            "wins": int(conference_wins[sim_idx, idx]),
            "losses": int(conference_losses[sim_idx, idx]),
            "ties": int(conference_ties[sim_idx, idx])
        }

    winner, criterion = nfl_tiebreakers.select_division_winner(
        DIVISIONS[division],
        final_records,
        results,
        div_records,
        conf_records
    )

    print(
        f"Simulation {sim_idx:5d} | "
        f"{division:9s} | "
        f"Winner: {str(winner):4s} | "
        f"Criterion: {criterion}"
    )

if not current_unresolved:
    print("No unresolved division tiebreakers to review.")


Rechecking 0 unresolved division tiebreakers for Week 9.
No unresolved division tiebreakers to review.


In [158]:
def independent_common_games(teams, game_results):

    opponents_by_team = {}

    for team in teams:
        opponents = set()

        for game in game_results:
            if game["home_team"] == team:
                opponents.add(game["away_team"])
            elif game["away_team"] == team:
                opponents.add(game["home_team"])

        opponents_by_team[team] = opponents

    # Exclude all tied teams
    common = set.intersection(
        *(opponents_by_team[team] for team in teams)
    ) - set(teams)

    records = {}

    for team in teams:
        wins = losses = ties = 0

        for game in game_results:
            home = game["home_team"]
            away = game["away_team"]
            winner = game["winner"]

            if team == home and away in common:
                opponent = away
            elif team == away and home in common:
                opponent = home
            else:
                continue

            if winner == team:
                wins += 1
            elif winner == opponent:
                losses += 1
            elif winner is None:
                ties += 1

        records[team] = {
            "wins": wins,
            "losses": losses,
            "ties": ties
        }

    return common, records


for teams in [
    ["BAL", "PIT", "CIN"],
    ["BAL", "PIT", "CIN", "CLE"]
]:

    common, records = independent_common_games(
        teams,
        season_results
    )

    for team in teams:
        model_record = nfl_tiebreakers.common_games_record(
            team,
            common,
            season_results
        )

        for stat in ["wins", "losses", "ties"]:
            assert model_record[stat] == records[team][stat]

    print(
        f"{len(teams)}-team common-games "
        f"calculation passed."
    )

3-team common-games calculation passed.
4-team common-games calculation passed.


In [159]:
# Check the total number of division selections
total_resolved = sum(division_win_counts.values())

expected_selections = N_SIMULATIONS * len(DIVISIONS)

assert total_resolved + len(unresolved) == expected_selections

print(f"Expected selections: {expected_selections:,}")
print(f"Resolved selections: {total_resolved:,}")
print(f"Unresolved selections: {len(unresolved)}")

# Check division probability totals
division_totals = (
    division_probabilities
    .groupby("division")["division_title_probability"]
    .sum()
)

print("\nDivision probability totals:")
print(division_totals.to_string())

# Verify that missing probability mass is explained
# by unresolved selections
unresolved_by_division = Counter(
    case["division"] for case in unresolved
)

for division, probability_sum in division_totals.items():
    expected_sum = (
        N_SIMULATIONS - unresolved_by_division[division]
    ) / N_SIMULATIONS

    assert abs(probability_sum - expected_sum) < 1e-10

print("\nAll division probability totals validated.")

Expected selections: 400,000
Resolved selections: 400,000
Unresolved selections: 0

Division probability totals:
division
AFC East     1.0
AFC North    1.0
AFC South    1.0
AFC West     1.0
NFC East     1.0
NFC North    1.0
NFC South    1.0
NFC West     1.0

All division probability totals validated.


In [160]:

# Compare current division-title probabilities
# against the previous week's final probabilities.

PREVIOUS_WEEK = TARGET_WEEK - 1

previous_output_dir = (
    PROJECT_ROOT
    / "rest_of_season"
    / "outputs"
    / f"week_{PREVIOUS_WEEK:02d}"
)

previous_path = (
    previous_output_dir
    / "division_title_probabilities_v2.csv"
)

assert previous_path.exists(), (
    f"Previous week's probabilities not found: {previous_path}"
)

previous = pd.read_csv(previous_path)

current = division_probabilities.copy()

comparison = previous[
    ["team", "division", "division_title_probability"]
].merge(
    current[
        ["team", "division", "division_title_probability"]
    ],
    on=["team", "division"],
    suffixes=("_previous", "_current"),
    validate="one_to_one"
)

assert len(comparison) == 32, (
    "Expected division-title probabilities for all 32 teams."
)

comparison["change_pp"] = (
    comparison["division_title_probability_current"]
    - comparison["division_title_probability_previous"]
) * 100

comparison = comparison.sort_values(
    "change_pp",
    key=abs,
    ascending=False
)

comparison.to_csv(
    ROS_OUTPUT_DIR / "division_title_probability_changes.csv",
    index=False
)

print(
    f"Largest division-title probability changes: "
    f"Week {PREVIOUS_WEEK} to Week {TARGET_WEEK}"
)

print(comparison.head(15).to_string(index=False))

print(
    "\nMaximum absolute change:",
    round(comparison["change_pp"].abs().max(), 4),
    "percentage points"
)

print(
    "\nSaved:",
    ROS_OUTPUT_DIR / "division_title_probability_changes.csv"
)


Largest division-title probability changes: Week 8 to Week 9
team  division  division_title_probability_previous  division_title_probability_current  change_pp
  TB NFC South                              0.26834                             0.51218     24.384
  KC  AFC West                              0.51108                             0.31764    -19.344
 DEN  AFC West                              0.48184                             0.66506     18.322
 MIN NFC North                              0.66454                             0.48326    -18.128
 ATL NFC South                              0.38672                             0.20606    -18.066
 PIT AFC North                              0.36946                             0.52080     15.134
 WAS  NFC East                              0.07544                             0.20648     13.104
 CHI NFC North                              0.10886                             0.21348     10.462
 JAX AFC South                              0.82

In [161]:
from collections import Counter

# Total number of division selections
expected = N_SIMULATIONS * len(DIVISIONS)
resolved = sum(division_win_counts.values())
unresolved_count = len(unresolved)

print(f"Expected:   {expected:,}")
print(f"Resolved:   {resolved:,}")
print(f"Unresolved: {unresolved_count:,}")

assert resolved + unresolved_count == expected

# Validate probabilities by division
for division, teams in DIVISIONS.items():

    division_titles = sum(
        division_win_counts[team]
        for team in teams
    )

    division_unresolved = sum(
        case["division"] == division
        for case in unresolved
    )

    assert (
        division_titles + division_unresolved
        == N_SIMULATIONS
    )

    probability_sum = division_titles / N_SIMULATIONS

    print(
        f"{division:10s} | "
        f"Probability total: {probability_sum:.5f} | "
        f"Unresolved: {division_unresolved}"
    )

print("\nAll division probability checks passed.")

Expected:   400,000
Resolved:   400,000
Unresolved: 0
AFC East   | Probability total: 1.00000 | Unresolved: 0
AFC North  | Probability total: 1.00000 | Unresolved: 0
AFC South  | Probability total: 1.00000 | Unresolved: 0
AFC West   | Probability total: 1.00000 | Unresolved: 0
NFC East   | Probability total: 1.00000 | Unresolved: 0
NFC North  | Probability total: 1.00000 | Unresolved: 0
NFC South  | Probability total: 1.00000 | Unresolved: 0
NFC West   | Probability total: 1.00000 | Unresolved: 0

All division probability checks passed.


In [162]:
import pandas as pd
import numpy as np

# Calculate expected final records across all simulations
expected_wins = simulated_wins.mean(axis=0)
expected_losses = simulated_losses.mean(axis=0)
expected_ties = simulated_ties.mean(axis=0)

# Calculate the most common final record for each team
most_likely_records = []

for team_idx, team in enumerate(team_names):

    records = pd.DataFrame({
        "wins": simulated_wins[:, team_idx],
        "losses": simulated_losses[:, team_idx],
        "ties": simulated_ties[:, team_idx]
    })

    most_common = (
        records
        .value_counts()
        .idxmax()
    )

    most_likely_records.append(
        f"{most_common[0]}-{most_common[1]}-{most_common[2]}"
    )

# Build the projected records table
projected_records = pd.DataFrame({
    "team": team_names,
    "expected_wins": expected_wins,
    "expected_losses": expected_losses,
    "expected_ties": expected_ties,
    "most_likely_record": most_likely_records
})

projected_records = projected_records.sort_values(
    "expected_wins",
    ascending=False
).reset_index(drop=True)

print(projected_records.to_string(index=False))

team  expected_wins  expected_losses  expected_ties most_likely_record
 BUF       12.71090          4.28910            0.0             13-4-0
  SF       12.24096          4.75904            0.0             12-5-0
 MIN       11.75672          5.24328            0.0             12-5-0
 DEN       11.75094          5.24906            0.0             12-5-0
 JAX       11.44182          5.55818            0.0             12-5-0
 PIT       11.27140          5.72860            0.0             11-6-0
 CIN       11.00636          5.99364            0.0             11-6-0
 CHI       10.97748          6.02252            0.0             11-6-0
 DET       10.72398          6.27602            0.0             11-6-0
  KC       10.62308          6.37692            0.0             11-6-0
  NE       10.03402          6.96598            0.0             10-7-0
 SEA        9.72990          7.27010            0.0             10-7-0
  GB        9.41014          7.58986            0.0              9-8-0
  LA  

In [163]:

def rank_division_teams(
    division_teams,
    final_records,
    game_results,
    division_records,
    conference_records
):
    """
    Rank all four teams in a division using
    the existing NFL division tiebreaker rules.

    Return None if a tie cannot be resolved.
    """

    remaining = list(division_teams)
    ranking = []

    while remaining:

        if len(remaining) == 1:
            ranking.append(remaining[0])
            break

        winner, criterion = (
            nfl_tiebreakers.select_division_winner(
                remaining,
                final_records,
                game_results,
                division_records,
                conference_records
            )
        )

        if winner is None:
            return None

        ranking.append(winner)
        remaining.remove(winner)

    return ranking


print("Division-ranking function loaded successfully.")


Division-ranking function loaded successfully.


In [164]:
from collections import Counter
import pandas as pd
import time

start_time = time.time()

# Track how often each team finishes in each position
finish_counts = {
    team: Counter()
    for team in team_names
}

# Track simulations where a full ranking cannot be resolved
unresolved_rankings = []

# Validate that first-place selections match the
# existing division winner calculation
champion_mismatches = []

for sim_idx in range(N_SIMULATIONS):

    results = nfl_tiebreakers.build_season_results(
        completed_games,
        future_games,
        home_wins,
        sim_idx
    )

    final_records = nfl_tiebreakers.calculate_final_records(
        results
    )

    div_records = {}
    conf_records = {}

    for team in team_names:
        idx = team_to_idx[team]

        div_records[team] = {
            "wins": int(division_wins[sim_idx, idx]),
            "losses": int(division_losses[sim_idx, idx]),
            "ties": int(division_ties[sim_idx, idx])
        }

        conf_records[team] = {
            "wins": int(conference_wins[sim_idx, idx]),
            "losses": int(conference_losses[sim_idx, idx]),
            "ties": int(conference_ties[sim_idx, idx])
        }

    for division, teams in DIVISIONS.items():

        ranking = rank_division_teams(
            teams,
            final_records,
            results,
            div_records,
            conf_records
        )

        if ranking is None:
            unresolved_rankings.append({
                "simulation": sim_idx,
                "division": division
            })
            continue

        # Check that the first-place team agrees with
        # our existing division winner function
        champion, criterion = (
            nfl_tiebreakers.select_division_winner(
                teams,
                final_records,
                results,
                div_records,
                conf_records
            )
        )

        if ranking[0] != champion:
            champion_mismatches.append({
                "simulation": sim_idx,
                "division": division,
                "ranking_winner": ranking[0],
                "division_winner": champion
            })

        for position, team in enumerate(ranking, start=1):
            finish_counts[team][position] += 1

    if (sim_idx + 1) % 10000 == 0:
        print(f"Completed {sim_idx + 1:,} simulations")

elapsed = time.time() - start_time

print(f"\nCompleted in {elapsed:.2f} seconds")
print(f"Unresolved division rankings: {len(unresolved_rankings)}")
print(f"Champion mismatches: {len(champion_mismatches)}")

Completed 10,000 simulations
Completed 20,000 simulations
Completed 30,000 simulations
Completed 40,000 simulations
Completed 50,000 simulations

Completed in 629.50 seconds
Unresolved division rankings: 7
Champion mismatches: 0


In [165]:
rows = []

for division, teams in DIVISIONS.items():
    for team in teams:

        rows.append({
            "team": team,
            "division": division,
            "first_place": finish_counts[team][1] / N_SIMULATIONS,
            "second_place": finish_counts[team][2] / N_SIMULATIONS,
            "third_place": finish_counts[team][3] / N_SIMULATIONS,
            "fourth_place": finish_counts[team][4] / N_SIMULATIONS
        })

division_finish_probabilities = pd.DataFrame(rows)

division_finish_probabilities = (
    division_finish_probabilities
    .sort_values(
        ["division", "first_place"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print(
    division_finish_probabilities.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

team  division  first_place  second_place  third_place  fourth_place
 BUF  AFC East       0.8403        0.1586       0.0011        0.0000
  NE  AFC East       0.1595        0.8308       0.0094        0.0003
 NYJ  AFC East       0.0001        0.0051       0.4918        0.5030
 MIA  AFC East       0.0001        0.0054       0.4978        0.4967
 PIT AFC North       0.5208        0.3389       0.1061        0.0342
 CIN AFC North       0.3924        0.3917       0.1583        0.0576
 CLE AFC North       0.0526        0.1680       0.4407        0.3388
 BAL AFC North       0.0342        0.1015       0.2950        0.5693
 JAX AFC South       0.9229        0.0698       0.0070        0.0002
 HOU AFC South       0.0662        0.6473       0.2584        0.0281
 IND AFC South       0.0104        0.2625       0.5856        0.1415
 TEN AFC South       0.0004        0.0204       0.1489        0.8302
 DEN  AFC West       0.6651        0.3037       0.0311        0.0001
  KC  AFC West       0.3176       

In [166]:
# Each fully ranked division must have one team
# in each finishing position.

for division, group in division_finish_probabilities.groupby("division"):

    unresolved_count = sum(
        case["division"] == division
        for case in unresolved_rankings
    )

    expected_total = (
        N_SIMULATIONS - unresolved_count
    ) / N_SIMULATIONS

    for column in [
        "first_place",
        "second_place",
        "third_place",
        "fourth_place"
    ]:

        actual_total = group[column].sum()

        assert abs(actual_total - expected_total) < 1e-10, (
            f"{division}: {column} total is incorrect"
        )

assert len(champion_mismatches) == 0

print("All division finish probability checks passed.")

All division finish probability checks passed.


In [167]:
title_cases = {
    (case["simulation"], case["division"])
    for case in unresolved
}

ranking_cases = {
    (case["simulation"], case["division"])
    for case in unresolved_rankings
}

print("Unresolved title cases:", len(title_cases))
print("Unresolved ranking cases:", len(ranking_cases))

print(
    "Title cases missing from ranking cases:",
    title_cases - ranking_cases
)

print(
    "Additional unresolved ranking cases:",
    ranking_cases - title_cases
)

Unresolved title cases: 0
Unresolved ranking cases: 7
Title cases missing from ranking cases: set()
Additional unresolved ranking cases: {(37131, 'NFC East'), (8758, 'NFC North'), (234, 'NFC East'), (36518, 'NFC East'), (20694, 'AFC South'), (13448, 'NFC North'), (48482, 'NFC East')}


In [168]:
# Combine projected records with division finish probabilities
final_division_standings = division_finish_probabilities.merge(
    projected_records,
    on="team",
    how="left"
)

# Arrange columns for readability
final_division_standings = final_division_standings[
    [
        "division",
        "team",
        "expected_wins",
        "expected_losses",
        "expected_ties",
        "most_likely_record",
        "first_place",
        "second_place",
        "third_place",
        "fourth_place"
    ]
]

# Sort by division and probability of winning division
final_division_standings = (
    final_division_standings
    .sort_values(
        ["division", "first_place"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

# Save outputs
division_finish_probabilities.to_csv(
    ROS_OUTPUT_DIR / "division_finish_probabilities.csv",
    index=False
)

final_division_standings.to_csv(
    ROS_OUTPUT_DIR / "final_division_standings.csv",
    index=False
)

final_division_standings.to_parquet(
    ROS_OUTPUT_DIR / "final_division_standings.parquet",
    index=False
)

print("Saved division finishing probabilities and final standings.")
print()
print(final_division_standings.to_string(index=False))

Saved division finishing probabilities and final standings.

 division team  expected_wins  expected_losses  expected_ties most_likely_record  first_place  second_place  third_place  fourth_place
 AFC East  BUF       12.71090          4.28910            0.0             13-4-0      0.84026       0.15864      0.00106       0.00004
 AFC East   NE       10.03402          6.96598            0.0             10-7-0      0.15948       0.83084      0.00940       0.00028
 AFC East  NYJ        4.87278         12.12722            0.0             5-12-0      0.00014       0.00508      0.49178       0.50300
 AFC East  MIA        4.41542         12.58458            0.0             4-13-0      0.00012       0.00544      0.49776       0.49668
AFC North  PIT       11.27140          5.72860            0.0             11-6-0      0.52080       0.33888      0.10608       0.03424
AFC North  CIN       11.00636          5.99364            0.0             11-6-0      0.39242       0.39168      0.15826       0.

In [169]:
# Preserve the original division-title probabilities
# rather than excluding simulations with unresolved lower-place ties.

division_title_lookup = dict(
    zip(
        division_probabilities["team"],
        division_probabilities["division_title_probability"]
    )
)

final_division_standings["first_place"] = (
    final_division_standings["team"].map(division_title_lookup)
)

division_finish_probabilities["first_place"] = (
    division_finish_probabilities["team"].map(division_title_lookup)
)

# Save the corrected outputs
division_finish_probabilities.to_csv(
    ROS_OUTPUT_DIR / "division_finish_probabilities.csv",
    index=False
)

final_division_standings.to_csv(
    ROS_OUTPUT_DIR / "final_division_standings.csv",
    index=False
)

final_division_standings.to_parquet(
    ROS_OUTPUT_DIR / "final_division_standings.parquet",
    index=False
)

print("Corrected first-place probabilities saved.")

Corrected first-place probabilities saved.
